# Phase 3 — Fine-tuning du correcteur ByT5

ByT5 travaille sur les octets : adapté aux différences sous-lexicales (e/é, chute de consonne, agglutination) d'une langue peu dotée.

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))
from src.config import CFG


## 1. Entraînement

La perte est une entropie croisée avec label smoothing (0.1). La MAE n'a pas de sens pour une sortie discrète de génération ; le label smoothing répond à l'intention réelle : éviter la sur-confiance sur des cibles synthétiques imparfaites.

In [ ]:
from src.phase3_train import build_parser, train
metrics = train(build_parser().parse_args(['--epochs', '4', '--batch-size', '8']))
metrics


## 2. WER avant/après et exemples qualitatifs

In [ ]:
print(f"WER avant : {metrics['wer_before']:.4f}")
print(f"WER après : {metrics['wer_after']:.4f}")
print(f"Réduction : {metrics['wer_reduction_rel']:.1%}")
for ex in metrics['examples'][:10]:
    print(ex, end='\n\n')


## 3. Essai libre

In [ ]:
from src.phase3_train import generate_batch, load_corrector
model, tok = load_corrector()
tests = ['mwen ap manje nan kay la', 'li te gen anpil bagay pou li fe']
for s, p in zip(tests, generate_batch(model, tok, tests)):
    print(f'{s}\n  -> {p}\n')


> Rappel : ces scores portent sur des corruptions **synthétiques**. Ils ne prédisent pas directement la performance sur de vraies sorties Whisper.